In [ ]:
import sys
from scapy.all import sniff
from scapy.utils import PcapWriter

# Создаем объект записи (append=True допишет данные, если файл уже существует)
pcap_writer = PcapWriter("captured_http.pcap", append=True, sync=True)


def packet_callback(packet):
    # Проверяем, содержит ли пакет слой с полезной нагрузкой (данными)
    if packet.haslayer("Raw"):
        try:
            # Извлекаем байты, декодируем в текст и игнорируем бинарные ошибки
            data = packet["Raw"].load.decode(errors="ignore")

            # Фильтруем только HTTP-запросы GET и POST
            if "GET" in data or "POST" in data:
                print(f"\n[Captured Packet] ->\n{data}")
                
                # Записываем оригинальный сетевой пакет в файл .pcap
                pcap_writer.write(packet)

        except Exception as e:
            print(f"Error decoding packet: {e}", file=sys.stderr)


def main():
    print("Starting packet sniffing on port 80... (Press Ctrl+C to stop)")
    try:
        # Запуск сниффера: фильтруем TCP-порт 80, не забиваем память (store=0)
        sniff(filter="tcp port 80", prn=packet_callback, store=0)
    except KeyboardInterrupt:
        print("\nSniffing stopped by user.")
    finally:
        # Обязательно закрываем файл при завершении работы скрипта
        pcap_writer.close()


if __name__ == "__main__":
    main()
